In [2]:
import math

deg2rad = lambda x : math.pi * x / 180.

LONG = [
  [
    [200, 500, 200, deg2rad(180), deg2rad(-5)],
    [500, 350, 100, deg2rad(0), deg2rad(-5)],
    [467.70, 449.56],
    "Long 1"
  ],[
    [100, -400, 100, deg2rad(30), deg2rad(0)],
    [500, -700, 0, deg2rad(150), deg2rad(0)],
    [649.52, 636.12],
    "Long 2"
  ],[
    [-200, 200, 250, deg2rad(240), deg2rad(15)],
    [ 500, 800,   0, deg2rad(45), deg2rad(15)],
    [1088.10, 1063.41],
    "Long 3"
  ],[
    [-300, 1200, 350, deg2rad(160), deg2rad(0)],
    [1000,  200,   0, deg2rad(30), deg2rad(0)],
    [1802.60, 1789.21],
    "Long 4"
  ],[
    [-500, -300, 600, deg2rad(150), deg2rad(10)],
    [1200,  900, 100, deg2rad(300), deg2rad(10)],
    [2245.14, 2216.40],
    "Long 5"
  ]
]

SHORT = [
  [
    [120, -30, 250, deg2rad(100), deg2rad(-10)],
    [220, 150, 100, deg2rad(300), deg2rad(-10)],
    [588.60, 583.47],
    "Short 1"
  ],[
    [380, 230, 200, deg2rad(30), deg2rad(0)],
    [280, 150,  30, deg2rad(200), deg2rad(0)],
    [667.71, 658.53],
    "Short 2"
  ],[
    [-80, 10, 250, deg2rad(20), deg2rad(0)],
    [ 50, 70,   0, deg2rad(240), deg2rad(0)],
    [979.34, 968.25],
    "Short 3"
  ],[
    [400, -250, 600, deg2rad(350), deg2rad(0)],
    [600, -150, 300, deg2rad(150), deg2rad(0)],
    [1169.73, 1161.55],
    "Short 4"
  ],[
    [-200, -200, 450, deg2rad(340), deg2rad(0)],
    [-300,  -80, 100, deg2rad(100), deg2rad(0)],
    [1367.56, 1354.12],
    "Short 5"
  ]
]

ADDITIONAL = [
    [
        [120, 40, 20, deg2rad(90), deg2rad(-5)],
        [300, 40, 15, deg2rad(-90), deg2rad(-5)],
        [],
        "Additional 1"
    ],
    [
        [120, 40, 20, deg2rad(90), deg2rad(-15)],
        [130, 120, 41, deg2rad(85), deg2rad(20)],
        [],
        "Additional 2"
    ]
]

INSIDE = [
    [
        [0, 0, 0, deg2rad(30), deg2rad(10)],
        [5, 10, 15, deg2rad(190), deg2rad(10)],
        [],
        "Inside pitch"
    ],
    [
        [0, 0, 0, deg2rad(-30), deg2rad(10)],
        [0, -30, 5, deg2rad(190), deg2rad(10)],
        [],
        "Inside yaw"
    ]
]

Rpitch = 40

In [3]:
# For all instances, we shift the origin to the start point, and rotate the coordinate frame so that the heading vector at the start point is along the positive z-axis. We note that the fourth and fifth entries in the vectors are the heading angle and the pitch angle, respectively.
import numpy as np
from math import cos as cos, sin as sin

instance_of_interest = "Inside yaw"

# We compute the modified initial and final location first
for instance in LONG + SHORT + ADDITIONAL + INSIDE:
    if instance[3] == instance_of_interest:
        
        # We shift the origin to the start point
        start = instance[0]
        end = instance[1]
        instance_mod = instance.copy()
        instance_mod[0] = np.array([0, 0, 0])
        instance_mod[1] = np.array([end[i] - start[i] for i in range(3)], dtype=float)

        print("Start and end location before modification:", start, end)
        print("Start and end location after modification:", instance_mod[0], instance_mod[1])

        # We compute the tangent vector at the final location
        heading_final_vect = np.array([math.cos(end[3]) * math.cos(end[4]), math.sin(end[3]) * math.cos(end[4]), math.sin(end[4])])
        x = heading_final_vect[0]; y = heading_final_vect[1]; z = heading_final_vect[2]
        psi = start[3]; phi = start[4]
        heading_final_vect_mod = np.array([x*cos(psi)*sin(phi) + y*sin(psi)*sin(phi) - z*cos(phi), -x*sin(psi) + y*cos(psi), x*cos(psi)*cos(phi) + y*sin(psi)*cos(phi) + z*sin(phi)])

        # Shrinking the problem by Rpitch so that the turning radius is one
        instance_mod[1][0] = instance_mod[1][0]/Rpitch
        instance_mod[1][1] = instance_mod[1][1]/Rpitch
        instance_mod[1][2] = instance_mod[1][2]/Rpitch

        # Rotating the final location as well in the new frame
        x = instance_mod[1][0]; y = instance_mod[1][1]; z = instance_mod[1][2]
        rotated_final_location = np.array([x*cos(psi)*sin(phi) + y*sin(psi)*sin(phi) - z*cos(phi), -x*sin(psi) + y*cos(psi), x*cos(psi)*cos(phi) + y*sin(psi)*cos(phi) + z*sin(phi)])

        print('Final location:', rotated_final_location, '\nFinal tangent vector in decimal:', heading_final_vect_mod, '\n')
        print("Norm of final tangent vector is ", np.linalg.norm(heading_final_vect_mod))

Start and end location before modification: [0, 0, 0, -0.5235987755982988, 0.17453292519943295] [0, -30, 5, 3.3161255787892263, 0.17453292519943295]
Start and end location after modification: [0 0 0] [  0. -30.   5.]
Final location: [-0.0579829  -0.64951905  0.39100893] 
Final tangent vector in decimal: [-0.30201139 -0.63302222 -0.71279169] 

Norm of final tangent vector is  0.9999999999999999


In [4]:
# We load the excel sheet containing the data for the paths
import pandas as pd
df = pd.read_excel("paths_from_analytic_CSC_modified.xlsx", sheet_name="Sheet1")
df

,Instance,Final configuration specification,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Path 1,Unnamed: 8,Unnamed: 9,...,Path 3,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Path 4,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26
0,NaN,xg,yg,zg,vx,vy,vz,phi1,theta1,d,...,phi1,theta1,d,phi2,theta2,phi1,theta1,d,phi2,theta2
1,Long 1,3.144155,3.75,-7.253571,0.173648,0,-0.984808,0.869158,2.77508,8.44862,...,0.8771,2.76603,7.88574,5.80552,0.294686,4.00534,3.90877,9.929,6.06742,5.6187
2,Long 2,2.5,-11.495191,4.910254,0,0.866025,-0.5,4.79912,1.1121,11.6398,...,5.03537,1.34764,11.927,3.87369,3.56538,1.62801,5.21556,13.3692,4.30954,3.08773
3,Long 3,0.410212,7.655445,-22.617215,-0.491481,0.25,-0.834234,4.67687,3.54744,24.7143,...,4.64001,3.55035,23.672,4.45205,0.553888,1.53477,2.89716,24.0586,4.75904,5.75877
4,Long 4,8.75,12.376661,-39.090514,0,-0.766044,-0.642788,0.936324,2.83026,42.4534,...,4.07617,3.54369,43.2312,5.77572,5.07037,0.974047,2.80544,40.6542,5.79479,1.16168
5,Long 5,8.523511,-47.230762,-23.645399,-0.319109,0.492404,-0.809758,4.88575,2.07599,53.5633,...,1.74387,4.26086,55.3916,6.04948,4.57944,4.8962,2.0352,51.614,6.05581,1.69341
6,Short 1,2.998868,-3.243436,4.587964,0.331707,-0.336824,-0.881204,2.32063,5.38924,8.07972,...,5.46187,1.01011,6.33149,3.15499,4.64394,2.30929,5.7818,5.78902,3.16373,2.14803
7,Short 2,4.25,-0.482051,-3.165064,0,0.173648,-0.984808,3.05415,4.24389,7.03882,...,2.99646,4.46378,4.91568,3.31857,1.35106,6.18245,2.59602,4.95961,3.45898,5.69651
8,Short 3,6.25,0.297973,3.567031,0,-0.642788,-0.766044,3.39782,5.42819,6.93426,...,0.237554,0.960818,5.43361,5.4263,2.16858,3.05067,5.17755,8.98906,5.55007,4.41664
9,Short 4,7.5,3.33026,4.489918,0,0.34202,-0.939693,3.60791,5.14776,11.1987,...,0.460926,1.26144,9.21374,3.45618,4.57073,3.49075,5.40391,9.07893,3.52426,2.10496


In [5]:
# We write a function for the homogeneous transformation matrix for the path
def homogeneous_transformation_matrix(theta, alpha, r, d):
    c_theta = np.cos(theta)
    s_theta = np.sin(theta)
    c_alpha = np.cos(alpha)
    s_alpha = np.sin(alpha)
    
    A = np.array([
        [c_theta, -s_theta*c_alpha,  s_theta*s_alpha, r*c_theta],
        [s_theta,  c_theta*c_alpha, -c_theta*s_alpha, r*s_theta],
        [0,        s_alpha,          c_alpha,         d        ],
        [0,        0,                0,               1        ]
    ])
    
    return A

In [6]:
# Function to compute the net homogeneous transformation matrix A_hand = A_1*A_2*A_3*A_4*A_5
def compute_net_transformation(theta_list, alpha_list, r_list, d_list):
    """
    Computes the net transformation matrix A_hand = A_1*A_2*A_3*A_4*A_5
    
    Parameters:
    theta_list: list of 5 theta values (one for each segment)
    alpha_list: list of 5 alpha values (one for each segment)
    r_list: list of 5 r values (one for each segment)
    d_list: list of 5 d values (one for each segment)
    
    Returns:
    A_hand: 4x4 net transformation matrix
    """
    # Start with identity matrix
    A_hand = np.eye(4)
    
    # Multiply all 5 transformation matrices
    for i in range(5):
        A_i = homogeneous_transformation_matrix(theta_list[i], alpha_list[i], r_list[i], d_list[i])
        A_hand = A_hand @ A_i

        # print("Ahand is ", A_hand)
    
    return A_hand

In [7]:
# Function to compute forward kinematics using DH parameters
def forward_kinematics_DH(theta_1, theta_2, d_3, theta_4, theta_5):
    """
    Computes forward kinematics using Denavit-Hartenberg parameters for RRPRR arm.
    
    DH Parameters (from Table I):
    Joint 1: revolute, r=1, alpha=π/2, d=0, theta=θ₁
    Joint 2: revolute, r=1, alpha=π/2, d=0, theta=θ₂
    Joint 3: prismatic, r=0, alpha=0, d=d₃, theta=0
    Joint 4: revolute, r=1, alpha=π/2, d=0, theta=θ₄
    Joint 5: revolute, r=1, alpha=π/2, d=0, theta=θ₅
    
    Parameters:
    theta_1, theta_2, theta_4, theta_5: joint angles (in radians)
    d_3: prismatic joint displacement
    
    Returns:
    A_hand: 4x4 net transformation matrix
    """
    # DH parameters based on Table I
    theta_list = [theta_1, theta_2, 0, theta_4, theta_5]
    alpha_list = [np.pi/2, np.pi/2, 0, np.pi/2, np.pi/2]
    r_list = [1, 1, 0, 1, 1]
    d_list = [0, 0, d_3, 0, 0]
    
    # Compute net transformation
    A_hand = compute_net_transformation(theta_list, alpha_list, r_list, d_list)
    
    return A_hand

In [8]:
# Function to get forward kinematics for a given instance and path from dataframe
def get_forward_kinematics_from_df(instance_name, path_number):
    """
    Retrieves Dubins path parameters from dataframe, converts to DH parameters,
    and computes forward kinematics.
    
    Conversion from Dubins to DH parameters (from equation 3):
    θ₁ = φ₁
    θ₂ = π - ψ₁
    d₃ = d
    θ₄ = φ₂ + π
    θ₅ = π - ψ₂
    
    Parameters:
    instance_name: name of the instance (e.g., "Short 1", "Long 1")
    path_number: path number in the dataframe
    
    Returns:
    A_hand: 4x4 net transformation matrix
    """
    # Get the row for the specific instance and path
    row = df[(df['Instance'] == instance_name)].iloc[0]
    
    # We consider in increments of five columns, starting frrom column number 1
    col_index = 7 + (path_number - 1) * 5

    # First column is phi_1, second column is psi_1, third column is d, fourth column is phi_2, fifth column is psi_2

    # Extract Dubins parameters (phi1, psi1, d, phi2, psi2)
    phi_1 = row.iloc[col_index]
    psi_1 = row.iloc[col_index + 1]
    d = row.iloc[col_index + 2]
    phi_2 = row.iloc[col_index + 3]
    psi_2 = row.iloc[col_index + 4]

    print("Dubins parameters are ", phi_1, psi_1, d, phi_2, psi_2)
    
    # Convert Dubins parameters to DH parameters
    theta_1 = phi_1
    theta_2 = np.pi - psi_1
    d_3 = d
    theta_4 = phi_2 - np.pi # If I use +np.pi, the net homogeneous transformation matrix does not match.
    theta_5 = np.pi - psi_2
    # theta_1 = phi_1
    # theta_2 = psi_1
    # d_3 = d
    # theta_4 = phi_2
    # theta_5 = psi_2
    
    # Compute forward kinematics
    A_hand = forward_kinematics_DH(theta_1, theta_2, d_3, theta_4, theta_5)

    # Computing the path length
    print("Path length is ", psi_1 + d + psi_2)
    
    return A_hand, psi_1 + d + psi_2

In [9]:
# Obtaining the parameters and displaying the matrix in floating point
get_forward_kinematics_from_df("Inside yaw", 1) # You can use this function and cross-check against the net homogeneous
# transformation matrix printed in the mathematica code provided by the authors.

Dubins parameters are  4.3931 4.44349 1.12458 0.100628 4.21008
Path length is  9.77815


(array([[-0.17936385, -0.93627841, -0.30201216, -0.05798539],
        [-0.69662366,  0.33764354, -0.63301842, -0.64951951],
        [ 0.69465393,  0.09684819, -0.71279474,  0.39100967],
        [ 0.        ,  0.        ,  0.        ,  1.        ]]),
 9.77815)

In [10]:
# Function to find the shortest path for each instance
def find_shortest_path_per_instance():
    """
    For each instance in the dataframe, finds the shortest valid path.
    A path is valid if all its parameters are finite.
    
    Returns:
    results: dictionary with instance names as keys and tuples of (path_number, path_length, A_hand) as values
    """
    results = {}
    
    # Get unique instances
    instances = df['Instance'].unique()
    
    for instance_name in instances:
        print(f"\n{'='*60}")
        print(f"Processing instance: {instance_name}")
        print(f"{'='*60}")
        
        shortest_path = None
        shortest_length = np.inf
        shortest_A_hand = None
        
        # Check up to 4 paths for each instance
        for path_num in range(1, 5):
            try:
                # Get the row for this instance
                row = df[df['Instance'] == instance_name].iloc[0]
                
                # Calculate column index for this path
                col_index = 7 + (path_num - 1) * 5
                
                # Extract path parameters
                phi_1 = row.iloc[col_index]
                psi_1 = row.iloc[col_index + 1]
                d = row.iloc[col_index + 2]
                phi_2 = row.iloc[col_index + 3]
                psi_2 = row.iloc[col_index + 4]
                
                # Check if all parameters are finite
                if all(np.isfinite([phi_1, psi_1, d, phi_2, psi_2])):
                    # Calculate path length
                    path_length = psi_1 + d + psi_2
                    
                    print(f"  Path {path_num}: length = {path_length:.4f}")
                    
                    # Check if this is the shortest so far
                    if path_length < shortest_length:
                        shortest_path = path_num
                        shortest_length = path_length
                        # Compute A_hand for this path
                        shortest_A_hand, _ = get_forward_kinematics_from_df(instance_name, path_num)
                else:
                    print(f"  Path {path_num}: INVALID (non-finite parameters)")
                    
            except (IndexError, KeyError):
                # Path doesn't exist for this instance
                print(f"  Path {path_num}: Does not exist")
                break
        
        # Store the result for this instance
        if shortest_path is not None:
            results[instance_name] = (shortest_path, shortest_length, shortest_A_hand)
            print(f"\n✓ Shortest path for {instance_name}: Path {shortest_path} with length {shortest_length:.4f}")
        else:
            print(f"\n✗ No valid paths found for {instance_name}")
    
    return results

# Execute the function
shortest_paths = find_shortest_path_per_instance()


Processing instance: nan
  Path 1: Does not exist

✗ No valid paths found for nan

Processing instance: Long 1
  Path 1: length = 17.2216
Dubins parameters are  0.869158 2.77508 8.44862 2.65096 5.99785
Path length is  17.22155
  Path 2: length = 13.2796
Dubins parameters are  4.02488 3.9569 8.6082 2.93531 0.714465
Path length is  13.279565000000002
  Path 3: length = 10.9465
Dubins parameters are  0.8771 2.76603 7.88574 5.80552 0.294686
Path length is  10.946456000000001
  Path 4: length = 19.4565

✓ Shortest path for Long 1: Path 3 with length 10.9465

Processing instance: Long 2
  Path 1: length = 15.7932
Dubins parameters are  4.79912 1.1121 11.6398 0.847016 3.04131
Path length is  15.793209999999998
  Path 2: length = 22.4926
  Path 3: length = 16.8400
  Path 4: length = 21.6725

✓ Shortest path for Long 2: Path 1 with length 15.7932

Processing instance: Long 3
  Path 1: length = 34.0059
Dubins parameters are  4.67687 3.54744 24.7143 1.34315 5.74417
Path length is  34.00591
  Pat

In [11]:
# Display summary of shortest paths
print("\n" + "="*80)
print("SUMMARY: SHORTEST PATHS FOR EACH INSTANCE")
print("="*80)
print(f"{'Instance':<20} {'Best Path':<12} {'Path Length':<15} {'Path Length * Rpitch':<15}")
print("-"*80)

for instance_name, (path_num, path_length, A_hand) in shortest_paths.items():
    print(f"{instance_name:<20} {path_num:<12} {path_length:<15.6f} {path_length*Rpitch:<15.6f} {path_length*Rpitch:<15.2f}")

print("="*80)


SUMMARY: SHORTEST PATHS FOR EACH INSTANCE
Instance             Best Path    Path Length     Path Length * Rpitch
--------------------------------------------------------------------------------
Long 1               3            10.946456       437.858240      437.86         
Long 2               1            15.793210       631.728400      631.73         
Long 3               2            26.479947       1059.197880     1059.20        
Long 4               4            44.621320       1784.852800     1784.85        
Long 5               4            55.342610       2213.704400     2213.70        
Short 1              2            7.483420        299.336800      299.34         
Short 2              2            7.048962        281.958480      281.96         
Short 3              3            8.563008        342.520320      342.52         
Short 4              2            10.556490       422.259600      422.26         
Short 5              2            10.936590       437.463600      4

In [12]:
# Generate 300 world-frame points (100 per CSC segment) for a given instance/path

def _get_instance_by_name(instance_name):
    for instance in LONG + SHORT + ADDITIONAL + INSIDE:
        if instance[3] == instance_name:
            return instance
    raise ValueError(f"Instance '{instance_name}' not found in LONG/SHORT/ADDITIONAL/INSIDE.")


def _sample_segment_points(p_start, p_end, n_points=100):
    t = np.linspace(0.0, 1.0, n_points)
    return (1.0 - t)[:, None] * p_start[None, :] + t[:, None] * p_end[None, :]


def _rotation_modified_to_world(initial_heading, initial_pitch):
    psi = initial_heading
    phi = initial_pitch

    # Rotation matrix from modified frame to world frame
    # Modified frame has z-axis along heading, xy-plane perpendicular to heading
    # This matrix's rows are projections of world axes onto modified frame
    # Or equivalently, columns are directions of modified axes in world frame
    R_world_to_modified = np.array([
        [np.cos(psi) * np.sin(phi), np.sin(psi) * np.sin(phi), -np.cos(phi)],
        [-np.sin(psi),               np.cos(psi),               0.0],
        [np.cos(psi) * np.cos(phi), np.sin(psi) * np.cos(phi),  np.sin(phi)]
    ])

    # To go from modified to world, we need the transpose (inverse of orthogonal matrix)
    return R_world_to_modified.T


def generate_points_for_instance_path(instance_name, path_number, points_per_segment=100):
    """
    Returns path points as an array of shape (300, 3) by default:
      - segment 1: origin -> translation of H2
      - segment 2: translation of H2 -> translation of H3
      - segment 3: translation of H3 -> translation of H5

    Then applies, in order:
      1) scale by Rpitch
      2) rotate to align with initial heading/pitch of the instance
      3) translate to the initial xyz location of the instance
    """
    if path_number < 1 or path_number > 4:
        raise ValueError("path_number must be in {1, 2, 3, 4}.")

    row_candidates = df[df['Instance'] == instance_name]
    if row_candidates.empty:
        raise ValueError(f"Instance '{instance_name}' not found in dataframe df.")
    row = row_candidates.iloc[0]

    col_index = 7 + (path_number - 1) * 5
    phi_1 = float(row.iloc[col_index])
    psi_1 = float(row.iloc[col_index + 1])
    d = float(row.iloc[col_index + 2])
    phi_2 = float(row.iloc[col_index + 3])
    psi_2 = float(row.iloc[col_index + 4])

    if not np.all(np.isfinite([phi_1, psi_1, d, phi_2, psi_2])):
        raise ValueError(f"Path {path_number} for instance '{instance_name}' is not finite.")

    # Same Dubins -> DH mapping used in get_forward_kinematics_from_df
    theta_1 = phi_1
    theta_2 = np.pi - psi_1
    d_3 = d
    theta_4 = phi_2 - np.pi
    theta_5 = np.pi - psi_2

    # theta_list = [theta_1, theta_2, 0.0, theta_4, theta_5]
    # alpha_list = [np.pi / 2, np.pi / 2, 0.0, np.pi / 2, np.pi / 2]
    # r_list = [1.0, 1.0, 0.0, 1.0, 1.0]
    # d_list = [0.0, 0.0, d_3, 0.0, 0.0]

    instance = _get_instance_by_name(instance_name)
    start = instance[0]
    start_xyz = np.array(start[:3], dtype=float)
    initial_heading = float(start[3])
    initial_pitch = float(start[4])

    p = np.array([])

    print("Start location is ", start_xyz, "and initial heading and pitch are ", initial_heading, initial_pitch)
    print("Final location before modification is ", instance[1][:3], "and final heading and pitch are ", instance[1][3], instance[1][4])

    # Considering d3 and psi_2 to be zero, and sampling psi_1 (segment 1)
    psi_1_samples = np.linspace(0, psi_1, points_per_segment)
    for i in psi_1_samples:
        H = forward_kinematics_DH(theta_1, np.pi - i, 0, -np.pi, np.pi)

        x = H[0, 3]; y = H[1, 3]; z = H[2, 3]
        # Modifying the points
        points_modified = _rotation_modified_to_world(initial_heading, initial_pitch) @ np.array([Rpitch*x, Rpitch*y, Rpitch*z]) + start_xyz
        # Appending points_modified to p
        p = np.vstack([p, points_modified.reshape(1, -1)]) if p.size else points_modified.reshape(1, -1)

    # Considering psi_2 to be zero, and sampling d_3 (segment 2)
    d_3_samples = np.linspace(0, d_3, points_per_segment)
    for i in d_3_samples:
        H = forward_kinematics_DH(theta_1, theta_2, i, -np.pi, np.pi)

        x = H[0, 3]; y = H[1, 3]; z = H[2, 3]
        # Modifying the points
        points_modified = _rotation_modified_to_world(initial_heading, initial_pitch) @ np.array([Rpitch*x, Rpitch*y, Rpitch*z]) + start_xyz
        p = np.vstack([p, points_modified.reshape(1, -1)]) if p.size else points_modified.reshape(1, -1)

    # Finally, sampling psi_2 (segment 3)
    psi_2_samples = np.linspace(0, psi_2, points_per_segment)
    for i in psi_2_samples:
        H = forward_kinematics_DH(theta_1, theta_2, d_3, theta_4, np.pi - i)

        x = H[0, 3]; y = H[1, 3]; z = H[2, 3]
        # Modifying the points
        points_modified = _rotation_modified_to_world(initial_heading, initial_pitch) @ np.array([Rpitch*x, Rpitch*y, Rpitch*z]) + start_xyz
        p = np.vstack([p, points_modified.reshape(1, -1)]) if p.size else points_modified.reshape(1, -1)

    # H = forward_kinematics_DH(theta_1, theta_2, d_3, theta_4, theta_5)

    # x = H[0, 3]; y = H[1, 3]; z = H[2, 3]
    # print("x, y, z are ", x, y, z)
    # points_modified = _rotation_modified_to_world(initial_heading, initial_pitch) @ np.array([Rpitch*x, Rpitch*y, Rpitch*z]) + start_xyz
    # print("Points modified is ", points_modified)
    # p = np.vstack([p, points_modified.reshape(1, -1)]) if p.size else points_modified.reshape(1, -1)

    return p
# Example:
# points_300x3 = generate_points_for_instance_path("Inside yaw", 2)
# points_300x3.shape

In [13]:
# Quick check
pts = generate_points_for_instance_path("Inside yaw", 1)
print(f"Shape: {pts.shape}")
print(f"First point: {pts[0]}")
print(f"Last point: {pts[-1]}")
print(pts)

Start location is  [0. 0. 0.] and initial heading and pitch are  -0.5235987755982988 0.17453292519943295
Final location before modification is  [0, -30, 5] and final heading and pitch are  3.3161255787892263 0.17453292519943295
Shape: (300, 3)
First point: [ 8.35570174e-15 -4.82416665e-15  1.70126148e-15]
Last point: [ 1.17648890e-06 -3.00000216e+01  5.00010312e+00]
[[ 8.35570174e-15 -4.82416665e-15  1.70126148e-15]
 [ 1.50965733e+00 -9.15766310e-01  3.24107280e-01]
 [ 2.97422250e+00 -1.89374018e+00  6.72467292e-01]
 [ 4.39074557e+00 -2.93195175e+00  1.04437837e+00]
 [ 5.75637336e+00 -4.02830986e+00  1.43909140e+00]
 [ 7.06835520e+00 -5.18060620e+00  1.85581134e+00]
 [ 8.32404848e+00 -6.38651981e+00  2.29369885e+00]
 [ 9.52092398e+00 -7.64362172e+00  2.75187191e+00]
 [ 1.06565709e+01 -8.94937986e+00  3.22940767e+00]
 [ 1.17287019e+01 -1.03011642e+01  3.72534428e+00]
 [ 1.27351574e+01 -1.16962518e+01  4.23868281e+00]
 [ 1.36739102e+01 -1.31318329e+01  4.76838929e+00]
 [ 1.45430694e+01 -

In [14]:
def export_path_to_csv(instance_name, path_number, points_per_segment=100, output_dir="."):
    """
    Export path points to a CSV file.
    
    Parameters:
    -----------
    instance_name : str
        Name of the instance (e.g., "Additional 1", "Long 1", "Short 1")
    path_number : int
        Path number (1-4)
    points_per_segment : int, optional
        Number of points per segment (default: 100)
    output_dir : str, optional
        Directory to save the CSV file (default: current directory)
    
    Returns:
    --------
    str : Path to the saved CSV file
    """
    import os
    
    # Generate the path points
    pts = generate_points_for_instance_path(instance_name, path_number, points_per_segment)
    
    # Create a DataFrame with x, y, z columns
    df_export = pd.DataFrame(pts, columns=['x', 'y', 'z'])
    
    # Create a clean filename from instance name (replace spaces with underscores)
    clean_instance_name = instance_name.replace(" ", "_")
    filename = f"{clean_instance_name}_path_{path_number}.csv"
    filepath = os.path.join(output_dir, filename)
    
    # Export to CSV
    df_export.to_csv(filepath, index=False)
    
    print(f"✓ Exported {len(pts)} points to: {filepath}")
    return filepath

# Example usage:
export_path_to_csv("Additional 2", 3)

Start location is  [120.  40.  20.] and initial heading and pitch are  1.5707963267948966 -0.2617993877991494
Final location before modification is  [130, 120, 41] and final heading and pitch are  1.4835298641951802 0.3490658503988659
✓ Exported 300 points to: .\Additional_2_path_3.csv


'.\\Additional_2_path_3.csv'